# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import sys
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from itertools import product
import geopandas as gpd
from pathlib import Path
import shutil
import json
from datetime import datetime
from tqdm import tqdm

root_path = Path(globals()['_dh'][0]).resolve().parent
sys.path.append(str(root_path))

from paths import config_path, input_path, api_path
from library.utilities import get_path
from library.randomizer import randomize_demand, random_factor_time_series
from library.growth import generate_growth_time_series
import library.scenario_constraints

In [2]:
# 2. Load the configuration

with (config_path / 'county-prototype.json').open('r', encoding='utf-8') as file:
    config = json.load(file)

## Set flag to clear the api folder
clear_api = True

In [3]:
# 3. Calculate the scenarios

## Extract and load the constraint functions
constraint_names = [ scenario['name'] for scenario in config['scenarios'] if scenario['type'] == 'constraint' ]
constraint_functions = [ getattr(library.scenario_constraints, name) for name in constraint_names ]

# Extract names and values from non-constraint parameters
parameter_objs = [obj for obj in config['scenarios'] if obj['type'] != 'constraint']
names = [obj['name'] for obj in parameter_objs]
values = [obj['values'] for obj in parameter_objs]

# Build default scenario
default_scenario = {
    obj["name"]: obj["default"]
    for obj in parameter_objs
    if "default" in obj
}

# Validate default scenario
if not all(fn(default_scenario) for fn in constraint_functions):
    raise ValueError(f"Default scenario violates constraints: {default_scenario}")

# Generate all scenarios
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Filter using constraints and mark the default
scenarios = []
for s in all_scenarios:
    if all(fn(s) for fn in constraint_functions):
        s_out = dict(s)
        if s == default_scenario:
            s_out["default"] = True
        scenarios.append(s_out)

In [4]:
# 3. Load the base demand

base_demand = pd.read_csv(input_path / config['loader']['properties']['access'] / 'base_demand' / get_path(config['loader']['name'],config['loader']['properties'], "csv"), index_col=['timestamp'], parse_dates=['timestamp'])

In [5]:
# 4 Execute transformations - Sort the transformations array

transformers = config['transformers']
transformers.sort(key=lambda x: x["order"])

In [6]:
# 4.1. Execute transformations - Split base demand (national) into geos

transformer = transformers[0]
inputs = transformer['inputs']
input = pd.read_csv(input_path / transformer['access'] / inputs[0]['path'], usecols=inputs[0]['columns'], dtype={inputs[0]['index']: str}, index_col=[inputs[0]['index']])

# Calculate geography demand
geography_demand_matrix = input[input.columns[0]].values[:, None] * base_demand[base_demand.columns[0]].values

# Randomize demand if the transformer has randomness set to true
if transformer['randomness']:
    geography_demand_matrix = randomize_demand(geography_demand_matrix, 0.1)

# Create new dataframe index (includes geography)
geography_index = pd.MultiIndex.from_product([input.index, base_demand.index], names=['geography', 'timestamp'])

# Create a new dataframe on the municipal demand with randomness applied
geography_demand = pd.DataFrame(geography_demand_matrix.ravel(), index=geography_index, columns=['demand'])

In [7]:
# 4.2. Execute transformations - Apply growth over time

## Extract geographies
geographies = geography_demand.index.get_level_values('geography').unique()

## Define growth parameters
start_time = pd.Timestamp(config['start-time'])
end_time = pd.Timestamp(config['end-time'])

new_timestamps = pd.date_range(start=start_time, end=end_time, freq=config['resolution'])

## Extract hourly demand patterns for 2024
hourly_demand_2024 = (
    geography_demand.loc[
        geography_demand.index.get_level_values('timestamp').year == 2024
    ]
    .reset_index()
)

# Repeat the 2024 demand pattern to match the new timestamps
hourly_demand_pattern = hourly_demand_2024.groupby("geography")["demand"].apply(
    lambda x: np.tile(x.values, len(new_timestamps) // len(x) + 1)[:len(new_timestamps)]
)

# Convert hourly demand pattern back to a 2D array (municipalities x timestamps)
base_demand_repeated = np.vstack(hourly_demand_pattern.values)

## Get the scenarios
transformer = transformers[1]
transformer_scenarios = transformer['scenarios']

## Working array
extended_data = []

for transformer_scenario in transformer_scenarios:
    scenario_name = transformer_scenario['name']
    scenario_index = transformer_scenario['index']
    if transformer_scenario['type'] == 'exp-growth-to-target':
        scenario_target = transformer_scenario['target']
    scenario_randomness = transformer_scenario.get('randomness', 0)  # Default to 0 if not provided

    ## Generate the index and growth factors for the extended demand
    growth_factors = generate_growth_time_series(new_timestamps, config['resolution'], transformer_scenario)

    # Generate randomness
    random_factors = np.random.uniform(-scenario_randomness, scenario_randomness, size=base_demand_repeated.shape)

    # Apply growth factors
    extended_demand_values = base_demand_repeated * growth_factors * (1+random_factors)

    new_index = pd.MultiIndex.from_product(
        [geographies, new_timestamps, [scenario_index]], names=["geography", "timestamp", scenario_name]
    )

    # Create the extended demand DataFrame
    extended_geography_demand = pd.DataFrame(
        data=extended_demand_values.flatten(),
        index=new_index,
        columns=["demand"]
    )

    # Append to the list
    extended_data.append(extended_geography_demand)

extended_geography_scenario_demand = pd.concat(extended_data)

In [8]:
# 4.3. Execute transformations - Split demand into ['industry', 'buildings', 'transport']

## TODO: Make this less naive

## This transform is not realistic. It does the following:
##      1. Assumes a flat industrial demand of 50% of the lowest day in July 2024 in each municipality
##      2. Calculates transport and buildings as respectively 5% and 95% of the remainder 

## Extract July data for each year
july_data = extended_geography_scenario_demand.loc[
    extended_geography_scenario_demand.index.get_level_values('timestamp').month == 7
]

# Compute the lowest July value per municipality, year, and growth scenario
lowest_july_per_year = (
    july_data
    .groupby([
        july_data.index.get_level_values('geography'),
        july_data.index.get_level_values('timestamp').year,
        july_data.index.get_level_values('growth')  # Ensure growth scenario is part of grouping
    ])['demand']
    .min()
)

# Convert to DataFrame and calculate industry demand
lowest_july_per_year = lowest_july_per_year.to_frame(name='lowest_july')
lowest_july_per_year['industry_demand'] = lowest_july_per_year['lowest_july'] * 0.5

# Create extended_geography_sector_demand and merge industry demand back with the full extended demand
extended_geography_scenario_sector_demand = extended_geography_scenario_demand.copy()

# Extract year from timestamp in the index
extended_geography_scenario_sector_demand['year'] = extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year

# Join the calculated industry demand on ['geography', 'year', 'growth']
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.join(
    lowest_july_per_year['industry_demand'],
    on=['geography', 'year', 'growth']
)

# Industry demand is constant across each municipality and year
extended_geography_scenario_sector_demand['industry'] = extended_geography_scenario_sector_demand['industry_demand']

# Compute the remainder for buildings and transport
extended_geography_scenario_sector_demand['remainder'] = (
    extended_geography_scenario_sector_demand['demand'] - extended_geography_scenario_sector_demand['industry']
)

# Split remainder into buildings (95%) and transport (5%)
extended_geography_scenario_sector_demand['buildings'] = extended_geography_scenario_sector_demand['remainder'] * 0.95
extended_geography_scenario_sector_demand['transport'] = extended_geography_scenario_sector_demand['remainder'] * 0.05

# Drop unnecessary intermediate columns
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.drop(columns=['industry_demand', 'remainder', 'year'])

# Rename 'demand' to 'total' for clarity
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.rename(columns={'demand': 'total'})

In [9]:
# 5. Write output

## Clear the api folder
if clear_api:
    shutil.rmtree(api_path)
    api_path.mkdir(parents=True, exist_ok=True)
    (api_path / ".gitignore").write_text("""
        # This folder contains the demand output
        # Therefore we ignore everything in this directory
        *
        # Except this file
        !.gitignore
    """, encoding='utf-8')

## Prepare data
print('Preparing data...')

geos = extended_geography_scenario_sector_demand.index.get_level_values('geography').unique()
years = extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year.unique()
growth_scenarios = extended_geography_scenario_sector_demand.index.get_level_values('growth').unique()

aggregations = config['output']['properties']['aggregations']

melted_aggregations = [
    {"resolution": item["resolution"], "aggregation": stat}
    for item in aggregations
    for stat in item["aggregation"]
]

## Write demand summed over geos, per resolution, aggregation, year, and growth scenario

print("Summing data over geographies...")
country_data = extended_geography_scenario_sector_demand.groupby(['timestamp', 'growth']).sum()

for year in tqdm(years, desc="Writing total geography data per year", unit="years"):
    yearly_data = country_data.loc[country_data.index.get_level_values('timestamp').year == year]
    for growth_scenario in growth_scenarios:
        # Filter data for the current growth scenario
        scenario_data = yearly_data.xs(growth_scenario, level='growth')
        for agg in melted_aggregations:
            if agg['aggregation'] != 'none':
                scenario_data.resample(agg['resolution']).agg(agg['aggregation']).to_csv(
                    api_path / f"demand_t,geography=00,resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year},growth={growth_scenario}.csv.gz",
                    compression='gzip'
                )
            else:
                scenario_data.to_csv(
                    api_path / f"demand_t,geography=00,resolution={agg['resolution']},sector=all,aggregation={'mean'},year={year},growth={growth_scenario}.csv.gz",
                    compression='gzip'
                )

## Group data by year and geography for all aggregations
print('Aggregating data per year...')
aggregations_1YE = next(item['aggregation'] for item in aggregations if item['resolution'] == '1YE')
grouped = extended_geography_scenario_sector_demand.groupby([extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year, 'geography', 'growth'])
yearly_stats = {stat: grouped.agg(stat) for stat in aggregations_1YE}

## Write all years per geography
for geo in tqdm(geos, desc="Progress (geos)", unit="geo"):
    for agg in aggregations_1YE:
        for growth_scenario in growth_scenarios:
            geo_data = yearly_stats[agg].xs((geo, growth_scenario), level=['geography', 'growth'])
            geo_data.to_csv(api_path / f"demand,geography={geo},resolution=1YE,sector=all,aggregation={agg},year=all,growth={growth_scenario}.csv.gz", compression='gzip')

## Write total geography per year and growth scenario
for year in tqdm(years, desc="Progress (years)", unit="years"):
    for agg in aggregations_1YE:
        for growth_scenario in growth_scenarios:
            year_data = yearly_stats[agg].loc[year]
            year_data_scenario = year_data.xs(growth_scenario, level='growth').copy()
            year_data_scenario.loc['00'] = year_data_scenario.agg(agg)
            year_data_scenario.to_csv(api_path / f"demand,geography=all,resolution=1YE,sector=all,aggregation={agg},year={year},growth={growth_scenario}.csv.gz", compression='gzip')

## Write all years for the total geography, per growth scenario
for agg in aggregations_1YE:
    for growth_scenario in growth_scenarios:
        national_data = yearly_stats['sum'].xs(growth_scenario, level='growth').groupby(level='timestamp').agg(agg)
        national_data.to_csv(api_path / f"demand,geography={'00'},resolution=1YE,sector=all,aggregation={agg},year=all,growth={growth_scenario}.csv.gz", compression='gzip')

## Write a single file for all geographies, years, and growth scenarios, per aggregation
year_data = yearly_stats['sum']
pd.concat([year_data, year_data.groupby(level=['timestamp', 'growth']).agg('sum').assign(geography='00').set_index('geography', append=True).reorder_levels(['timestamp', 'geography', 'growth']).sort_index()]).sort_index().to_csv(
    api_path / f"demand,geography=all,resolution=1YE,sector=all,aggregation={agg},year=all,growth=all.csv.gz", compression='gzip')

# Write globals
globals ={
    'upper_bound': year_data['total'].values.max(),
    'lower_bound': year_data['total'].values.min()
}

(api_path / "globals.json").write_text(json.dumps(globals, indent=4, ensure_ascii=False), encoding='utf-8')

## Write demand per geo and year

def process_geo_year_growth(task):
    geo, year, growth_scenario = task  # Unpack the task tuple
    geography_data = extended_geography_scenario_sector_demand.xs(geo, level='geography')
    yearly_data = geography_data.loc[geography_data.index.get_level_values('timestamp').year == year]
    scenario_data = yearly_data.xs(growth_scenario, level='growth')
    
    for agg in melted_aggregations:
        if agg['aggregation'] != 'none':
            scenario_data.resample(agg['resolution']).agg(agg['aggregation']).to_csv(
                api_path / f"demand_t,geography={geo},resolution={agg['resolution']},sector=all,aggregation={agg['aggregation']},year={year},growth={growth_scenario}.csv.gz",
                compression='gzip'
            )
        else:
            scenario_data.to_csv(
                api_path / f"demand_t,geography={geo},resolution={agg['resolution']},sector=all,aggregation={config['loader']['properties']['aggregation']},year={year},growth={growth_scenario}.csv.gz",
                compression='gzip'
            )

# Create tasks as (geo, year) pairs
tasks = list(product(geos, years, growth_scenarios))

# Use ProcessPoolExecutor with a top-level function
with ProcessPoolExecutor() as executor:
    list(tqdm(executor.map(process_geo_year_growth, tasks), total=len(tasks), desc="Geo-Year Pairs"))

# Write the geojson file (no energy data in it right now. Just a renaming.)

shutil.copy(input_path / config['access'] / config['geography']['source'], api_path / f"geo,division={config['geography']['type']}.geojson")

# Write the parameters.json

print("Writing parameters.json")

geographies_gdp = gpd.read_file(input_path / config['access'] / config['geography']['source'], encoding='utf-8')
geographies = geographies_gdp[['geo_id', 'geo_name', 'geo_type']].copy()

## Add Sweden as a whole
new_row = pd.DataFrame({
    "geo_type": ["country"],
    "geo_id": ["00"],
    "geo_name": ["Sverige"]
})

geographies = pd.concat([geographies, new_row], ignore_index=True)

# Replace "none" aggregation with base aggregation for base resolution
aggregations_parameters = [
    {
        "resolution": item["resolution"],
        "aggregation": config['loader']['properties']['aggregation'] if item["resolution"] == config['loader']['properties']['resolution'] else item["aggregation"]
    }
    for item in config['output']['properties']['aggregations']
]

# Write parameters.json
parameters = {
    'years': list(range(pd.Timestamp(config['start-time']).year, pd.Timestamp(config['end-time']).year)),
    'geography-type': config['geography']['type'],
    'geographies': geographies.to_dict(orient="records"),
    'aggregations': aggregations_parameters,
    'sectors': config['output']['properties']['sectors'],
    'scenarios': {
        config['transformers'][1]['scenarios'][0]['name']: [{'index': scenario['index'], 'label': scenario['label']} for scenario in config['transformers'][1]['scenarios']]
    }
}

(api_path / "parameters.json").write_text(json.dumps(parameters, indent=4, ensure_ascii=False), encoding='utf-8')

# Write scenarios.json
(api_path / "scenarios.json").write_text(json.dumps(scenarios, indent=4, ensure_ascii=False), encoding='utf-8')

# Write config.json
(api_path / "config.json").write_text(json.dumps(config, indent=4, ensure_ascii=False), encoding='utf-8')


Preparing data...
Summing data over geographies...


Writing total geography data per year: 100%|██████████| 21/21 [00:11<00:00,  1.76years/s]


Aggregating data per year...


Geo-Year Pairs: 100%|██████████| 1323/1323 [01:54<00:00, 11.56it/s]


Writing parameters.json


6089